In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset, random_split

import matplotlib.pyplot as plt
import numpy as np
import os

from milliontrees.datasets.TreePolygons import TreePolygonsDataset
from segmentation_pipeline import fit_segmentation



In [2]:
torch.cuda.empty_cache()

In [3]:
dataset = TreePolygonsDataset(
    root_dir="./data",
    version="0.12",
    #download=True,
    mini=True,
    include_unsupervised=True,
    split_scheme="random",
    
)

[MillionTrees] Loaded TreePolygons v0.12 (mini)
[MillionTrees] Data dir: data/TreePolygons_v0.12
[MillionTrees] Split: random | Images train/test/total: 42/6/48
[MillionTrees] Annotations: 12286 | Sources selected/available: 16/16
[MillionTrees] exclude_sources: ['*unsupervised*']


In [4]:
from collections import Counter
from PIL import Image

image_dir ="./data/TreePolygons_v0.12/images"
mask_dir = "./data/TreePolygons_v0.12/masks"

img_sizes = []
mask_sizes = []

# images
for filename in os.listdir(image_dir):
    path = os.path.join(image_dir, filename)
    try:
        with Image.open(path) as img:
            img_sizes.append(img.size)
    except:
        continue

# masks
for filename in os.listdir(mask_dir):
    path = os.path.join(mask_dir, filename)
    try:
        with Image.open(path) as img:
            mask_sizes.append(img.size)
    except:
        continue

print("Image sizes:", Counter(img_sizes))
print("Mask sizes:", Counter(mask_sizes))

Image sizes: Counter({(1500, 1500): 9, (2000, 2000): 9, (3000, 3000): 6, (1000, 1000): 6, (2048, 2048): 3, (400, 400): 3, (500, 500): 3, (960, 540): 3, (801, 800): 2, (1555, 1946): 1, (752, 741): 1, (801, 801): 1, (810, 794): 1})
Mask sizes: Counter({(1500, 1500): 9, (2000, 2000): 9, (3000, 3000): 6, (1000, 1000): 6, (400, 400): 3, (500, 500): 3, (2048, 2048): 3, (960, 540): 3, (801, 800): 2, (1555, 1946): 1, (752, 741): 1, (801, 801): 1, (810, 794): 1})


In [5]:
print("Same :", Counter(img_sizes) == Counter(mask_sizes))

Same : True


In [6]:
train_full = dataset.get_subset("train")
test_full = dataset.get_subset("test")

# split train → train/val
n_train = int(0.8 * len(train_full))
n_val = len(train_full) - n_train

train_base, val_base = random_split(
    train_full,
    [n_train, n_val],
    generator=torch.Generator().manual_seed(42)
)

In [7]:
import psutil
import torch

print("RAM usage:", psutil.virtual_memory().percent, "%")

if torch.cuda.is_available():
    print("GPU memory (before):", torch.cuda.memory_allocated() / 1e9, "GB")


RAM usage: 73.2 %


In [8]:
def resize_and_pad(image, mask, target_size=224):
    # image: (C, H, W)
    # mask: (H, W)

    _, h, w = image.shape

    # calcul du scale (ratio conservé)
    scale = min(target_size / h, target_size / w)

    new_h = int(h * scale)
    new_w = int(w * scale)

    # resize
    image = F.interpolate(
        image.unsqueeze(0),
        size=(new_h, new_w),
        mode="bilinear",
        align_corners=False
    ).squeeze(0)

    mask = F.interpolate(
        mask.unsqueeze(0).unsqueeze(0).float(),
        size=(new_h, new_w),
        mode="nearest"
    ).squeeze(0).squeeze(0).long()

    #padding pour atteindre target_size
    pad_h = target_size - new_h
    pad_w = target_size - new_w

    #padding: (left, right, top, bottom)
    image = F.pad(image, (0, pad_w, 0, pad_h))
    mask = F.pad(mask, (0, pad_w, 0, pad_h))

    return image, mask

In [9]:
class TreePolygonsBinaryDataset(Dataset):
    def __init__(self, base_dataset):
        self.base = base_dataset

    def __len__(self):
        return len(self.base)

    def __getitem__(self, idx):
        #metadata, image, targets = self.base[idx]
        data = self.base[idx]
        if len(data) == 3:
            metadata, image, targets = data
            instance_masks = targets["y"]
            binary_mask = instance_masks.any(dim=0).long()
        elif len(data) == 2:
            image, binary_mask = data
        else:
            raise ValueError("Unexpected dataset output format")
            
        instance_masks = targets["y"]  # [N, H, W]
        binary_mask = instance_masks.any(dim=0).long()  # [H, W]
    
        # probleme de tailles differnetes d'images
        image, binary_mask = resize_and_pad(image, binary_mask, 224)
    
        return image, binary_mask
    
    
# wrap binary
train_ds = TreePolygonsBinaryDataset(train_base)
val_ds = TreePolygonsBinaryDataset(val_base)
test_ds = TreePolygonsBinaryDataset(test_full)


BATCH_SIZE = 2

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    pin_memory=False
)
val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    pin_memory=False
)
test_loader = DataLoader(
    test_ds,
    batch_size=BATCH_SIZE,
    pin_memory=False
)


for loader_name, loader in [("train", train_loader), ("val", val_loader)]:
    print(f"iterating {loader_name}...")
    for images, masks in loader:
        print(f"  {loader_name}: images={images.shape}, masks={masks.shape}")
        break

iterating train...
  train: images=torch.Size([2, 3, 224, 224]), masks=torch.Size([2, 224, 224])
iterating val...
  val: images=torch.Size([2, 3, 224, 224]), masks=torch.Size([2, 224, 224])


In [10]:
# Sees if everything works so far
# Just iterate once, no model computed

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
for loader_name, loader in [("train", train_loader), ("val", val_loader)]:
    print(f"iterating {loader_name}...")
    for images, masks in loader:
        # ------------------------------------------------------------------------ Uncomment to see
        # print(f"  {loader_name}: images={images.shape}, masks={masks.shape}")
        # ------------------------------------------------------------------------

        # break after one batch to see if it reaches here
        break

iterating train...
iterating val...


for name, ds in [("train", train_ds), ("val", val_ds), ("test", test_ds)]:
    print(f"\nChecking {name} dataset...")
    for i in range(min(5, len(ds))):
        image, mask = ds[i]
        print(f"  {name} sample {i}:")
        print(f"    image.shape = {image.shape}")
        print(f"    mask.shape  = {mask.shape}")
        print(f"    mask.dtype  = {mask.dtype}")
        print(f"    mask.unique = {mask.unique().tolist()}")

In [11]:
class DinoV2Segmentation(nn.Module):
    def __init__(self, backbone, feat_dim=384):
        super().__init__()
        self.backbone = backbone

        self.decoder = nn.Sequential(
            nn.Conv2d(feat_dim, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(128, 2, kernel_size=1)
        )

    def forward(self, x):
        feats = self.backbone.forward_features(x)
        patch_tokens = feats["x_norm_patchtokens"]

        B, N, C = patch_tokens.shape
        W = int(N ** 0.5)
        H = W
        feat_map = patch_tokens.permute(0, 2, 1).reshape(B, C, H, W)

        logits = self.decoder(feat_map)
        logits = F.interpolate(logits, size=(224, 224), mode="bilinear", align_corners=False)

        return logits


In [12]:
def calculate_iou(preds, masks, num_classes=2):
    # preds: [B, C, H, W], masks: [B, H, W]
    preds = torch.argmax(preds, dim=1)
    
    #se concentre sur la classe 1 (les arbres)
    intersection = ((preds == 1) & (masks == 1)).float().sum((1, 2))
    union = ((preds == 1) | (masks == 1)).float().sum((1, 2))
    
    iou = (intersection + 1e-6) / (union + 1e-6)
    return iou.mean().item()

## Load DINOv2

In [13]:
backbone = torch.hub.load("facebookresearch/dinov2", "dinov2_vits14")

for param in backbone.parameters():
    param.requires_grad = False


Using cache found in /Users/zahra/.cache/torch/hub/facebookresearch_dinov2_main
/Users/zahra/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/Users/zahra/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/Users/zahra/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


## Training setup

In [14]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
model = DinoV2Segmentation(backbone).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer,
    step_size=5,
    gamma=0.5
)


cpu


In [15]:
def train_segmentation(model, train_loader, val_loader, criterion, optimizer, scheduler, device, epochs):
    history = []

    for epoch in range(epochs):
        model.train()
        train_loss, train_iou = 0, 0

        for images, masks in train_loader:
            images, masks = images.to(device), masks.to(device)
            optimizer.zero_grad()

            logits = model(images)
            loss = criterion(logits, masks)
            
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            train_iou += calculate_iou(logits, masks)

        scheduler.step()

        # Validation
        model.eval()
        val_loss, val_iou = 0, 0
        with torch.no_grad():
            for images, masks in val_loader:
                images, masks = images.to(device), masks.to(device)
                logits = model(images)
                
                val_loss += criterion(logits, masks).item()
                val_iou += calculate_iou(logits, masks)

        avg_train_loss = train_loss / len(train_loader)
        avg_val_loss = val_loss / len(val_loader)
        avg_val_iou = val_iou / len(val_loader)

        print(f"Epoch {epoch+1}/{epochs}")
        print(f"  Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")
        print(f"  Val IoU: {avg_val_iou:.4f}")

        history.append({
            "train_loss": avg_train_loss,
            "val_loss": avg_val_loss,
            "val_iou": avg_val_iou
        })

    return history

In [ ]:
history = train_segmentation(
    model,
    train_loader,
    val_loader,
    criterion,
    optimizer,
    scheduler,
    device,
    epochs=1
)


In [ ]:
train_losses = [h["train_loss"] for h in history]
val_losses = [h["val_loss"] for h in history]

plt.plot(train_losses, label="train")
plt.plot(val_losses, label="val")
plt.legend()
plt.title("Loss curves")
plt.show()


In [ ]:
os.makedirs("runs/dinov2_segmentation", exist_ok=True)

torch.save(
    model.state_dict(),
    "runs/dinov2_segmentation/best_model_dinov2_segmentation.pth"
    "runs/dinov2_segmentation/1_epoch_dinov2_segmentation.pth"

    
)


# pipeline

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
model = DinoV2Segmentation(backbone).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

results, trained_model = fit_segmentation(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    test_loader=test_loader,
    criterion=criterion,
    optimizer=optimizer,
    device=device,
    run_dir="./runs/dinov2_segmentation",
    epochs=15,
    scheduler=scheduler,
    patience=3,
)

In [ ]:
print('\n'*100)